In [15]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("app1").master("local[*]") \
     .config("spark.sql.adaptive.enabled", "false") \
      .getOrCreate()

df=spark.read.format("csv") \
   .option("inferSchema",True) \
    .option("header",True) \
    .load("orders.csv")

df.show(5)

+-------+----------+--------+--------+-------+-------------------+
|OrderID|  Customer|    Item|Quantity|  Price|          OrderDate|
+-------+----------+--------+--------+-------+-------------------+
|  O1001|Customer_4| Monitor|       3| 1901.7|2025-08-01 11:11:00|
|  O1002|Customer_2|Keyboard|       1|1314.52|2025-08-01 01:10:00|
|  O1003|Customer_2| Monitor|       1| 498.44|2025-08-01 02:27:00|
|  O1004|Customer_1|   Mouse|       2| 691.07|2025-08-01 00:59:00|
|  O1005|Customer_5|  Laptop|       5|1491.08|2025-08-01 02:51:00|
+-------+----------+--------+--------+-------+-------------------+
only showing top 5 rows



In [3]:
import pyspark

print(pyspark.__version__)

3.5.8


In [18]:
# Q1. 

"""Given employer history data, where each record contains details about an employee’s 4
work history — employer, job position, and the start and end dates of each job.

We need to find out how many users had Microsoft as their employer, 
and immediately after that, they started working at Google, with no other 
employers between these two positions."""

# first create the dataframe
from pyspark.sql.functions import *
from pyspark.sql import Window

linkedin_data = [
    (1, 'Microsoft', 'developer', '2020-04-13', '2021-11-01'),
    (1, 'Google', 'developer', '2021-11-01', None),
    (2, 'Google', 'manager', '2021-01-01', '2021-01-11'),
    (2, 'Microsoft', 'manager', '2021-01-11', None),
    (3, 'Microsoft', 'analyst', '2019-03-15', '2020-07-24'),
    (3, 'Amazon', 'analyst', '2020-08-01', '2020-11-01'),
    (3, 'Google', 'senior analyst', '2020-11-01', '2021-03-04'),
    (4, 'Google', 'junior developer', '2018-06-01', '2021-11-01'),
    (4, 'Google', 'senior developer', '2021-11-01', None),
    (5, 'Microsoft', 'manager', '2017-09-26', None),
    (6, 'Google', 'CEO', '2015-10-02', None)
]

linkedn_schema= [
  'emp_id', 
  'employer', 
  'position', 
  'start_date', 
  'end_date'
]

employee_data=spark.createDataFrame(linkedin_data,schema=linkedn_schema)

window_spec= Window.partitionBy("emp_ID").orderBy(col("start_date").asc())
employee_switch=employee_data.withColumn("next_employee",lead("employer",1).over(window_spec))
final_employee_data=employee_switch.filter((lower(col("employer"))=='microsoft') & (lower(col("next_employee"))=='google'))
final_employee_data.show()
final_employee_data.explain(True)
num_partitions=final_employee_data.rdd.getNumPartitions()
print(num_partitions)

+------+---------+---------+----------+----------+-------------+
|emp_id| employer| position|start_date|  end_date|next_employee|
+------+---------+---------+----------+----------+-------------+
|     1|Microsoft|developer|2020-04-13|2021-11-01|       Google|
+------+---------+---------+----------+----------+-------------+

== Parsed Logical Plan ==
'Filter ((lower('employer) = microsoft) AND (lower('next_employee) = google))
+- Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531, next_employee#537]
   +- Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531, next_employee#537, next_employee#537]
      +- Window [lead(employer#528, 1, null) windowspecdefinition(emp_ID#527L, start_date#530 ASC NULLS FIRST, specifiedwindowframe(RowFrame, 1, 1)) AS next_employee#537], [emp_ID#527L], [start_date#530 ASC NULLS FIRST]
         +- Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531]
            +- LogicalRDD [emp_i

In [25]:
## Q1 using the spark sql
employee_data.createOrReplaceTempView("employee_table")

sql_df=spark.sql("select * from (select *, lead(employer) over(partition by emp_id order by end_date) as next_employer from employee_table) as " \
"tmp where employer='Microsoft' and next_employer='Google'")

sql_df.show()
sql_df.explain(True)

+------+---------+--------+----------+--------+-------------+
|emp_id| employer|position|start_date|end_date|next_employer|
+------+---------+--------+----------+--------+-------------+
|     2|Microsoft| manager|2021-01-11|    NULL|       Google|
+------+---------+--------+----------+--------+-------------+

== Parsed Logical Plan ==
'Project [*]
+- 'Filter (('employer = Microsoft) AND ('next_employer = Google))
   +- 'SubqueryAlias tmp
      +- 'Project [*, 'lead('employer) windowspecdefinition('emp_id, 'end_date ASC NULLS FIRST, unspecifiedframe$()) AS next_employer#692]
         +- 'UnresolvedRelation [employee_table], [], false

== Analyzed Logical Plan ==
emp_id: bigint, employer: string, position: string, start_date: string, end_date: string, next_employer: string
Project [emp_id#527L, employer#528, position#529, start_date#530, end_date#531, next_employer#692]
+- Filter ((employer#528 = Microsoft) AND (next_employer#692 = Google))
   +- SubqueryAlias tmp
      +- Project [emp_i

#### Question 2

In [ ]:
"""
Given a table of hotels with various attributes (hotel_address, 
additional_number_of_scoring, review_date, average_score, hotel_name, 
reviewer_nationality, negative_review, review_total_negative_word_counts, 
total_number_of_reviews, positive_review, review_total_positive_word_counts, 
total_number_of_reviews_reviewer_has_given, reviewer_score, tags, days_since_review, lat, lng ), 
We need to find the top 10 hotels with the highest average scores. The output should include:

The hotel name.
The average score of the hotel.
The records should be sorted by average score in descending order.

"""

# first create the dataframe 
data = [
  ('123 Ocean Ave, Miami, FL', 3, '2024-11-10', 4.2, 'Ocean View', 'American', 'Room small, but clean.', 5, 150, 'Great location and friendly staff!', 8, 30, 4.5, 'beachfront, family-friendly', '5 days', 25.7617, -80.1918),   
  ('456 Mountain Rd, Boulder, CO', 2, '2024-11-12', 3.9, 'Mountain Lodge', 'Canadian', 'wifi slow.', 3, 120, 'nice rooms.', 10, 20, 4.0, 'scenic, nature', '3 days', 40.015, -105.2705),  
  ('789 Downtown St, New York, NY', 5, '2024-11-15', 4.7, 'Central Park Hotel', 'British', 'Noisy, sleep.', 7, 200, 'Perfect location near Central Park.', 12, 50, 4.7, 'luxury, city-center', '1 day', 40.7831, -73.9712),
  ('101 Lakeside Blvd, Austin, TX', 1, '2024-11-08', 4.0, 'Lakeside Inn', 'Mexican', 'food avg.', 4, 80, 'Nice, friendly service.', 6, 15, 3.8, 'relaxing, family', '10 days', 30.2672, -97.7431),
  ('202 River Ave, Nashville, TN', 4, '2024-11-13', 4.5, 'Riverside', 'German', 'Limited parking', 2, 175, 'Great rooms.', 9, 25, 4.2, 'riverfront, peaceful', '2 days', 36.1627, -86.7816)
]
# Define columns for the hotel DataFrame
schema_columns = [
  "hotel_address", 
  "additional_number_of_scoring", 
  "review_date", 
  "customer_score", 
  "hotel_name",            
  "reviewer_nationality", 
  "negative_review", 
  "review_total_negative_word_counts", 
  "total_number_of_reviews",           
  "positive_review", 
  "review_total_positive_word_counts", 
  "total_number_of_reviews_reviewer_has_given",
  "reviewer_score", 
  "tags", 
  "days_since_review", 
  "lat", 
  "lng"
]
hotel_df=spark.createDataFrame(data,schema_columns)
hotel_df.show()



+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+
|       hotel_address|additional_number_of_scoring|review_date|customer_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|
+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------

In [32]:
## find out the avg score using the addtional_number_scoring+customer_score+reviewer_score/total_number_of_reviews_reviewer_has_given

hotel_review_score= hotel_df.withColumn("avg_score",round((col("additional_number_of_scoring")+col("customer_score")*col("reviewer_score"))/col("total_number_of_reviews_reviewer_has_given"),2))
hotel_review_score.show()

+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+---------+
|       hotel_address|additional_number_of_scoring|review_date|customer_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|avg_score|
+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+--------------------------

In [ ]:
## highest avg score hotel
highest_review=hotel_review_score.orderBy(col('avg_score').desc())
highest_review.show()

+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+-------+---------+---------+
|       hotel_address|additional_number_of_scoring|review_date|customer_score|        hotel_name|reviewer_nationality|     negative_review|review_total_negative_word_counts|total_number_of_reviews|     positive_review|review_total_positive_word_counts|total_number_of_reviews_reviewer_has_given|reviewer_score|                tags|days_since_review|    lat|      lng|avg_score|
+--------------------+----------------------------+-----------+--------------+------------------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+--------------------------

#### Question 3